# 📊 Statistical Evaluation Metrics for Machine Learning

**A comprehensive, hands-on guide to the metrics that every data scientist needs to know.**

This notebook covers:
- **Classification Metrics:** Confusion Matrix, Accuracy, Precision, Recall, F1 Score, Specificity, ROC-AUC, Log Loss
- **Regression Metrics:** MAE, MSE, RMSE, R²
- **When to use which metric** — a practical decision guide
- **Python code** with scikit-learn for every metric

---

## 📑 Table of Contents

1. [Confusion Matrix](#1)
2. [Accuracy](#2)
3. [Precision](#3)
4. [Recall (Sensitivity)](#4)
5. [F1 Score](#5)
6. [Specificity](#6)
7. [ROC Curve & AUC](#7)
8. [Log Loss](#8)
9. [Regression Metrics (MAE, MSE, RMSE, R²)](#9)
10. [Choosing the Right Metric](#10)
11. [Full Python Implementation](#11)

---

In [ ]:
# Required imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, log_loss,
    mean_absolute_error, mean_squared_error, r2_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ All imports ready!')

## 🔧 Generate Sample Data

We'll create a synthetic binary classification dataset to demonstrate all metrics consistently.

In [ ]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, weights=[0.7, 0.3],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train a simple model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

<a id='1'></a>
# 1. 📐 Confusion Matrix

The **Confusion Matrix** is the foundation of all classification metrics. It summarizes a model's correct and incorrect predictions in a **2×2** table:

|  | **Predicted Positive** | **Predicted Negative** |
|---|:---:|:---:|
| **Actual Positive** | ✅ TP (True Positive) | ⚠️ FN (False Negative) |
| **Actual Negative** | ❌ FP (False Positive) | 🟢 TN (True Negative) |

### Key Terms

| Term | Description | Error Type |
|------|-------------|------------|
| **TP** | Actually positive → correctly predicted positive | — |
| **FP** | Actually negative → incorrectly predicted positive | **Type I Error** |
| **FN** | Actually positive → incorrectly predicted negative | **Type II Error** |
| **TN** | Actually negative → correctly predicted negative | — |

In [ ]:
# Compute and visualize confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['Actual Neg', 'Actual Pos'],
            ax=axes[0], linewidths=2, linecolor='white',
            annot_kws={'size': 18, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix', fontsize=16, fontweight='bold')

# Component breakdown
components = ['TP', 'TN', 'FP', 'FN']
values = [tp, tn, fp, fn]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
bars = axes[1].bar(components, values, color=colors, edgecolor='white', linewidth=2)
axes[1].set_title('Component Breakdown', fontsize=16, fontweight='bold')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

print(f'\n📊 TP={tp} | TN={tn} | FP={fp} | FN={fn}')
print(f'   Total samples: {tp + tn + fp + fn}')

<a id='2'></a>
# 2. 📏 Accuracy

The ratio of **correct predictions** to total predictions.

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

> ⚠️ **Accuracy Paradox:** With imbalanced datasets, a model that *always* predicts the majority class can achieve high accuracy while being completely useless. For example, if 99% of emails are not spam, a model that labels everything "not spam" gets 99% accuracy — but catches zero spam!

In [ ]:
acc = accuracy_score(y_test, y_pred)
acc_manual = (tp + tn) / (tp + tn + fp + fn)

print(f'Accuracy (sklearn) : {acc:.4f} ({acc*100:.1f}%)')
print(f'Accuracy (manual)  : {acc_manual:.4f} ({acc_manual*100:.1f}%)')
print(f'\nFormula: ({tp} + {tn}) / ({tp} + {tn} + {fp} + {fn}) = {tp+tn}/{tp+tn+fp+fn}')

<a id='3'></a>
# 3. 🎯 Precision

Of everything the model labeled **positive**, how much was actually positive?

$$\text{Precision} = \frac{TP}{TP + FP}$$

### 🔑 When does Precision matter?

When **false positives are costly**:
- **Spam filter** — blocking an important email is harmful
- **Search engine** — irrelevant results degrade user experience
- **Recommendation system** — wrong suggestions erode trust

In [ ]:
prec = precision_score(y_test, y_pred)
prec_manual = tp / (tp + fp)

print(f'Precision (sklearn) : {prec:.4f} ({prec*100:.1f}%)')
print(f'Precision (manual)  : {prec_manual:.4f} ({prec_manual*100:.1f}%)')
print(f'\nFormula: {tp} / ({tp} + {fp}) = {tp}/{tp+fp}')
print(f'\n→ Out of {tp+fp} positive predictions, {tp} were correct.')

<a id='4'></a>
# 4. 🔍 Recall (Sensitivity / True Positive Rate)

Of all actual positives, how many did the model catch?

$$\text{Recall} = \frac{TP}{TP + FN}$$

### 🔑 When does Recall matter?

When **false negatives are costly**:
- **Cancer screening** — missing a patient can be life-threatening
- **Fraud detection** — missed fraud causes major financial loss
- **Security systems** — undetected threats are dangerous

### ⚖️ Precision–Recall Trade-off

> Lowering the decision threshold → more positives predicted → **Recall ↑** but **Precision ↓**  
> Raising the decision threshold → fewer positives predicted → **Precision ↑** but **Recall ↓**

In [ ]:
rec = recall_score(y_test, y_pred)
rec_manual = tp / (tp + fn)

print(f'Recall (sklearn) : {rec:.4f} ({rec*100:.1f}%)')
print(f'Recall (manual)  : {rec_manual:.4f} ({rec_manual*100:.1f}%)')
print(f'\nFormula: {tp} / ({tp} + {fn}) = {tp}/{tp+fn}')
print(f'\n→ Out of {tp+fn} actual positives, {tp} were detected and {fn} were missed.')

<a id='5'></a>
# 5. ⚖️ F1 Score

The **harmonic mean** of Precision and Recall — a single number that balances both.

$$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

> 💡 **Why harmonic mean?** The harmonic mean punishes extreme imbalances. If Precision = 1.0 but Recall = 0.01, the arithmetic mean would be 0.505, but the harmonic mean (F1) is only 0.02 — correctly flagging the poor performance.

### F-Beta Score

$$F_\beta = (1 + \beta^2) \times \frac{\text{Precision} \times \text{Recall}}{(\beta^2 \times \text{Precision}) + \text{Recall}}$$

| Score | β | Emphasis | Use Case |
|-------|---|----------|----------|
| **F0.5** | 0.5 | Precision | Spam filters, search engines |
| **F1** | 1.0 | Equal | General classification |
| **F2** | 2.0 | Recall | Medical diagnosis, security |

In [ ]:
from sklearn.metrics import fbeta_score

f1 = f1_score(y_test, y_pred)
f1_manual = 2 * (prec * rec) / (prec + rec)
f05 = fbeta_score(y_test, y_pred, beta=0.5)
f2 = fbeta_score(y_test, y_pred, beta=2)

print(f'F1 Score (sklearn) : {f1:.4f}')
print(f'F1 Score (manual)  : {f1_manual:.4f}')
print(f'\nF0.5 Score (Precision-weighted): {f05:.4f}')
print(f'F2   Score (Recall-weighted)   : {f2:.4f}')

# Visualize all classification metrics
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 
           'F1': f1, 'F0.5': f05, 'F2': f2}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db', '#9b59b6', '#f39c12', '#2ecc71', '#1abc9c', '#e67e22']
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, 
              edgecolor='white', linewidth=2)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('📊 Classification Metrics Overview', fontsize=16, fontweight='bold')
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend()
plt.tight_layout()
plt.show()

<a id='6'></a>
# 6. 🛡️ Specificity (True Negative Rate)

How well does the model identify **true negatives**?

$$\text{Specificity} = \frac{TN}{TN + FP}$$

| Metric | Focuses On | Question |
|--------|-----------|----------|
| **Sensitivity (Recall)** | Positive class | How well do we detect the *sick*? |
| **Specificity** | Negative class | How well do we identify the *healthy*? |

> Together, Sensitivity and Specificity form the basis of the **ROC curve**.

In [ ]:
specificity = tn / (tn + fp)

print(f'Specificity : {specificity:.4f} ({specificity*100:.1f}%)')
print(f'Sensitivity : {rec:.4f} ({rec*100:.1f}%)')
print(f'\nFormula: {tn} / ({tn} + {fp}) = {tn}/{tn+fp}')

<a id='7'></a>
# 7. 📈 ROC Curve & AUC

### ROC Curve
The **ROC (Receiver Operating Characteristic)** curve plots **TPR (Recall)** vs **FPR (1 − Specificity)** across all classification thresholds.

$$\text{TPR} = \frac{TP}{TP + FN} \qquad \text{FPR} = \frac{FP}{FP + TN}$$

### AUC (Area Under the Curve)
AUC summarizes the ROC curve into a single number (0 to 1).

| AUC | Rating | Meaning |
|-----|--------|----------|
| **1.0** | Perfect | Flawless separation |
| **0.9–1.0** | Excellent | High discriminative power |
| **0.7–0.9** | Good | Acceptable |
| **0.5–0.7** | Poor | Barely better than random |
| **0.5** | Random | Coin flip |

> 🔑 **AUC is threshold-independent** — ideal for comparing models and handling imbalanced data.

In [ ]:
auc = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#e94560', linewidth=2.5, label=f'Model (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='#e94560')
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=16, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])

# Threshold vs Metrics
precisions = []
recalls = []
f1s = []
thresh_range = np.linspace(0.1, 0.9, 50)
for t in thresh_range:
    y_t = (y_prob >= t).astype(int)
    if y_t.sum() > 0 and (1 - y_t).sum() > 0:
        precisions.append(precision_score(y_test, y_t, zero_division=0))
        recalls.append(recall_score(y_test, y_t, zero_division=0))
        f1s.append(f1_score(y_test, y_t, zero_division=0))
    else:
        precisions.append(0)
        recalls.append(0)
        f1s.append(0)

axes[1].plot(thresh_range, precisions, label='Precision', linewidth=2)
axes[1].plot(thresh_range, recalls, label='Recall', linewidth=2)
axes[1].plot(thresh_range, f1s, label='F1 Score', linewidth=2, linestyle='--')
axes[1].set_xlabel('Decision Threshold', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Precision–Recall Trade-off', fontsize=16, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

print(f'\n🏆 AUC Score: {auc:.4f}')

<a id='8'></a>
# 8. 📉 Log Loss (Logarithmic Loss)

Evaluates predicted **probabilities**, not just class labels. Lower = better.

$$\text{Log Loss} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \cdot \log(\hat{y}_i) + (1 - y_i) \cdot \log(1 - \hat{y}_i)\right]$$

- **Perfect model:** Log Loss = 0
- **Random guessing (0.5):** Log Loss ≈ 0.693

> 🔑 Frequently used as a **Kaggle competition** scoring metric. A model can have high accuracy but poor log loss if its probability estimates are poorly calibrated.

In [ ]:
ll = log_loss(y_test, y_prob)
ll_random = log_loss(y_test, np.full_like(y_prob, 0.5))

print(f'Log Loss (Model)  : {ll:.4f}')
print(f'Log Loss (Random) : {ll_random:.4f}')
print(f'\n→ The model is {ll_random/ll:.1f}x better than random guessing.')

# Visualize probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, label='Actual Negative', color='#3498db')
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, label='Actual Positive', color='#e74c3c')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Predicted Probability Distribution', fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

<a id='9'></a>
# 9. 📊 Regression Metrics

For models that predict **continuous values** instead of classes.

---

### MAE — Mean Absolute Error
Average of absolute differences. **Robust to outliers.**

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}|y_i - \hat{y}_i|$$

### MSE — Mean Squared Error
Average of squared differences. **Penalizes large errors more.**

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

### RMSE — Root Mean Squared Error
Square root of MSE — **same scale as target**. Most widely used.

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}$$

### R² — Coefficient of Determination
How much variance the model explains. **Closer to 1 = better.**

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

| R² | Interpretation |
|-----|----------------|
| **1.0** | Perfect fit |
| **0.7–1.0** | Good |
| **0.4–0.7** | Moderate |
| **< 0.4** | Weak |
| **< 0** | Worse than mean |

In [ ]:
# Generate regression data
np.random.seed(42)
y_true_reg = np.array([3.0, 5.0, 2.5, 7.0, 4.5, 6.2, 1.8, 8.3, 3.7, 5.9])
y_pred_reg = y_true_reg + np.random.normal(0, 0.5, len(y_true_reg))

mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)

print('📊 Regression Metrics')
print('=' * 35)
print(f'MAE  : {mae:.4f}')
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_true_reg, y_pred_reg, s=100, c='#e94560', edgecolors='white', linewidth=2, zorder=5)
lims = [min(y_true_reg.min(), y_pred_reg.min()) - 0.5, 
        max(y_true_reg.max(), y_pred_reg.max()) + 0.5]
axes[0].plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Values', fontsize=12)
axes[0].set_ylabel('Predicted Values', fontsize=12)
axes[0].set_title(f'Actual vs Predicted (R² = {r2:.3f})', fontsize=14, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_true_reg - y_pred_reg
axes[1].bar(range(len(residuals)), residuals, color=['#2ecc71' if r >= 0 else '#e74c3c' for r in residuals],
            edgecolor='white', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].set_xlabel('Sample Index', fontsize=12)
axes[1].set_ylabel('Residual (Actual − Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

<a id='10'></a>
# 10. 🧭 Choosing the Right Metric

| Scenario | Best Metric | Why? |
|----------|-------------|------|
| Balanced dataset | **Accuracy, F1** | Accuracy is reliable |
| Imbalanced dataset | **F1, AUC, Precision/Recall** | Accuracy is misleading |
| FP is costly | **Precision** | Minimize false alarms |
| FN is costly | **Recall** | Don't miss positives |
| Probability quality | **Log Loss, AUC** | Evaluates confidence |
| Model comparison | **AUC** | Threshold-independent |
| Continuous prediction | **RMSE, MAE, R²** | Regression problems |

<a id='11'></a>
# 11. 🐍 Full Classification Report

In [ ]:
print('=' * 60)
print('   📊 COMPLETE MODEL EVALUATION REPORT')
print('=' * 60)

print(f'\n🎯 Accuracy    : {acc:.4f}')
print(f'🎯 Precision   : {prec:.4f}')
print(f'🔍 Recall      : {rec:.4f}')
print(f'⚖️  F1 Score    : {f1:.4f}')
print(f'🛡️  Specificity : {specificity:.4f}')
print(f'📈 AUC-ROC     : {auc:.4f}')
print(f'📉 Log Loss    : {ll:.4f}')

print(f'\n{"=" * 60}')
print('   📋 DETAILED CLASSIFICATION REPORT')
print(f'{"=" * 60}\n')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# 📚 Summary Cheat Sheet

| Metric | Formula | Range | Best When |
|--------|---------|-------|-----------|
| **Accuracy** | (TP+TN) / Total | 0–1 | Balanced classes |
| **Precision** | TP / (TP+FP) | 0–1 | FP is costly |
| **Recall** | TP / (TP+FN) | 0–1 | FN is costly |
| **F1 Score** | 2·P·R / (P+R) | 0–1 | Balance P & R |
| **Specificity** | TN / (TN+FP) | 0–1 | Negative class focus |
| **AUC** | Area under ROC | 0–1 | Model comparison |
| **Log Loss** | −Σ[y·log(ŷ)] | 0–∞ | Probability quality |
| **MAE** | Σ\|y−ŷ\| / N | 0–∞ | Outlier-robust regression |
| **RMSE** | √(Σ(y−ŷ)²/N) | 0–∞ | General regression |
| **R²** | 1 − SS_res/SS_tot | −∞–1 | Variance explained |

---

<div style='text-align:center; padding:20px;'>
    <h3>📊 Statistical Evaluation Metrics</h3>
    <p>A comprehensive resource for Machine Learning & Data Science.</p>
    <p>If you found this useful, please <strong>upvote</strong> 👍</p>
</div>